<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F04_ray_on_vertex.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 04 · Ray on Vertex AI

Run the **Python-runtime models on a Ray-on-Vertex cluster** in parallel with the in-BigQuery native track — the same `main.run(cfg)` entrypoint as every other demo, dispatched to Ray by `cfg.python_runtime="ray"`. The cluster autoscales per pool by default — its initial size is a deterministic function of the run's fan-out — the job runs on it, and the cluster is torn down in a `finally` so nothing bills after the run.

> **Runs from any authenticated client** — local workstation or an in-GCP kernel. Job submission goes through the cluster's dashboard proxy host (`*.aiplatform-training.googleusercontent.com`), which serves the `JobSubmissionClient` handshake as long as the cluster is provisioned on a **PSC-I network attachment** with a **dashboard-capable head node** (`n1-standard-16`+). Both are wired by the Terraform network module and defaulted in `config` — so the older "must run inside GCP / `524` from outside" caveat no longer applies.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable. The `[ray]` extra must be installed in the kernel (`pip install -e 'scale-forecasting[ray]'`).

In [1]:
# Cloud bootstrap: clone + install the LOCKED dependency set (uv.lock) so
# `import scale_forecasting` resolves against the exact versions every other surface runs.
# We add the [ray] extra (the Vertex Ray client, version-matched to the cluster). Uses uv
# (Colab ships it; we install it if missing) to install the frozen lock into a PRIVATE prefix,
# then puts that prefix + src/ first on sys.path. Harmless locally — if it already imports, no-op.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

EXTRAS = ["ray"]

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    uv = shutil.which("uv")
    if uv is None:  # Colab Enterprise ships uv; install it if this runtime doesn't
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or "uv"
    # Resolve the checked-in lock to a requirements file (no re-resolve), then install EXACTLY
    # that set into this kernel — the same versions the container + packed-venv are built from.
    reqs = os.path.abspath(os.path.join(REPO_DIR, "colab-requirements.txt"))
    extra_flags = [f for e in EXTRAS for f in ("--extra", e)]
    subprocess.run(
        [uv, "export", "--frozen", "--no-emit-project", "--no-hashes", "--no-dev", *extra_flags,
         "-o", reqs],
        cwd=REPO_DIR, check=True,
    )
    # Install the LOCKED set into a PRIVATE directory (not the runtime's system site-packages),
    # then put it FIRST on sys.path. Managed images (Colab Enterprise, Vertex) ship numpy 2.x that
    # can't be cleanly downgraded in place: uv skips the version-satisfied packages it can't fully
    # uninstall, so numpy-2 .so files end up beside numpy-1 python and `import numpy` dies with
    # "dtype size changed, Expected 96 ... got 88". Installing to a separate prefix and SHADOWING the
    # base packages (never touching them) gives this kernel a clean, self-consistent numpy 1.26.4 —
    # the same set every other surface runs — with zero risk of a mixed install.
    target = os.path.abspath(os.path.join(REPO_DIR, ".colab-deps"))
    subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "--target", target, "-r", reqs],
        check=True,
    )
    # src/ carries our package; `target` carries its locked deps. Both go ahead of the base image's
    # site-packages so imports resolve to the versions we installed, not the runtime's.
    sys.path.insert(0, target)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment + Ray infra (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default.

The Ray submitter needs a little more (beyond that identity), resolved by `RayInfra`: **`SF_COMPUTE_SA`** and **`SF_CODE_BUCKET`** are required. Connectivity is picked in precedence order: **`SF_RAY_NETWORK_ATTACHMENT`** (PSC-I — the supported path) → `SF_RAY_NETWORK` (VPC peering) → unset (public). Every value comes straight from `terraform output` in `terraform/main` — `RayInfra.from_terraform_outputs()` reads them all, including `network_attachment_id`.

In [2]:
from google.cloud import bigquery

from scale_forecasting.ray_submit import RayInfra
from scale_forecasting.settings import Settings

settings = Settings.resolve()
infra = RayInfra.resolve()  # raises naming the first missing SF_COMPUTE_SA / SF_CODE_BUCKET
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref

# Which connectivity mode did RayInfra pick? PSC-I (network_attachment) is the supported path.
_mode = "PSC-I" if infra.network_attachment else "VPC-peering" if infra.network else "public"
print("deployment:", DATASET, "region:", settings.region)
print("ray infra: compute_sa set:", bool(infra.compute_sa), "| code_bucket:", infra.code_bucket)
print("ray connectivity:", _mode, "| attachment:", infra.network_attachment or "(none)")

deployment: statmike-scale-forecasting.scale_forecasting region: us-central1
ray infra: compute_sa set: True | code_bucket: statmike-scale-forecasting-code
ray connectivity: PSC-I | attachment: projects/307701787156/regions/us-central1/networkAttachments/scale-forecasting-ray


## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Ray / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [3]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record*. No separate JSON file to open.

This run mixes both tracks under **one `run_id`**: the stats models run on **Ray** (`compute_engine='ray'`) while the natives run in **BigQuery** (`compute_engine='bigquery'`), in parallel — the Ray analogue of notebook 03's Spark ∥ BigQuery.

> **Coming from PySpark + pandas UDFs?** This is the *same* per-`(series, model)` unit of work you'd run in an `applyInPandas` fan-out — `worker.run_cell` is identical on Spark and Ray — and it reads the **same** `source_series_iceberg` input. The difference: no Spark session to stand up or tune, Ray ∥ BigQuery under one `run_id`, and one Ray cluster can host many frameworks (including Spark workloads via RayDP). The engine reads the source panel two ways, chosen by `compute.ray_read_mode`: the default `driver_collect` (the BigQuery Storage Read API client) and the opt-in `ray_data` (`ray.data.read_bigquery`) — both go through the same Storage Read API, which is what makes the Iceberg/native input transparent. *Next steps (not yet shipped):* distributing the `ray.data` read as blocks straight into the fan-out (skipping the driver round-trip) and Spark-on-Ray (RayDP).

- **`RUN_NAME`** carries a timestamp so each execution is its own clean run (the cell tables are append-only, so a fresh `run_id` avoids overwriting a still-buffering prior run).
- **`SOURCE_TABLE`** defaults to `source_series_iceberg` to *demonstrate Ray reading the managed-Iceberg input*; the example also ships as `source_series_native` (identical series) so you can flip the suffix and compare storage formats on the same run shape.
- **`RAY_MODELS`** run on the cluster; **`BQ_MODELS`** run in BigQuery. The shipped example is univariate; the generic exog seam stays available for your own data (set `features.exog` + carry the column in your source table).
- **`RAY_READ_MODE`** picks the driver's source reader: `"driver_collect"` (default, the proven Storage Read client) or `"ray_data"` (the Ray-native `ray.data.read_bigquery`). Both read the same table over the same Storage Read API and hand the fan-out an identical panel — the knob just selects the client.
- **`USE_GPU=False`** → `plan_cluster` sizes the GPU pool to **zero** (CPU workers only, no T4 quota needed). Flip to `True` and add `"neuralprophet"` to `RAY_MODELS` for the fractional-T4 showpiece — identical call, one flag apart (needs `NVIDIA_T4_GPUS` quota in a `RAY_REGIONS` region).

> Expect **~15–25 min**: cluster stand-up dominates. `RAY_REGIONS` lets the launcher hop US regions if one transiently stocks out on capacity.

In [4]:
from scale_forecasting.config import RunConfig
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb04 ray+bq {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = (
    "source_series_iceberg"  # Ray reads the SAME Iceberg input; _native to compare storage
)
RAY_MODELS = ["theta", "holtwinters"]  # → run on the Ray cluster
BQ_MODELS = ["arima_plus", "timesfm"]  # → run in BigQuery, in parallel
HORIZON = 28
SERIES_LIMIT = 100  # the demo scale (same first 100 series every approach uses)
HOLIDAYS = ["US"]
USE_GPU = False  # True + "neuralprophet" in RAY_MODELS → T4 showpiece
RAY_READ_MODE = "driver_collect"  # or "ray_data" (ray.data.read_bigquery) — same Storage Read API
BACKTEST = True  # OOF metric panel comparable across both engines
N_FOLDS = 2
RAY_REGIONS = ["us-central1", "us-east1", "us-west1"]
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="ray",  # dispatch the Python models to Ray (not Spark)
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=[*RAY_MODELS, *BQ_MODELS],  # router splits these by runtime automatically
    features={"holidays": HOLIDAYS},
    backtest={"enabled": BACKTEST, "n_folds": N_FOLDS, "horizon": HORIZON, "step": HORIZON},
    compute={"use_gpu": USE_GPU, "ray_regions": RAY_REGIONS, "ray_read_mode": RAY_READ_MODE},
)
run_id = make_run_id(cfg)
print("run_id:", run_id)
print("ray models:", RAY_MODELS, "| bq models:", BQ_MODELS, "| runtime:", cfg.python_runtime)

run_id: nb04-ray-bq-1789918508-344fc058e7f8
ray models: ['theta', 'holtwinters'] | bq models: ['arima_plus', 'timesfm'] | runtime: ray


## Run — Ray ∥ BigQuery, one `run_id`

One call. `main.run(cfg)` sizes + creates the Ray cluster on the PSC-I attachment — autoscaling per pool by default, its initial size a deterministic function of the fan-out — runs `RAY_MODELS` on it **in parallel** with `BQ_MODELS` in BigQuery under the shared `run_id`, then tears the cluster down in a `finally`. It returns the same `run_id` we computed above.

In [5]:
from scale_forecasting import main

returned = main.run(cfg)
assert returned == run_id
print("ray ∥ bigquery run complete:", run_id)

2026-09-20 15:35:32,183 WARNING scale_forecasting.ray_cluster: preflight rules out us-east1 without attempting a create: the PSC-I network attachment is in us-central1; network attachments are regional and there is none in us-east1


2026-09-20 15:35:32,184 WARNING scale_forecasting.ray_cluster: preflight rules out us-west1 without attempting a create: the PSC-I network attachment is in us-central1; network attachments are regional and there is none in us-west1


[Ray on Vertex AI]: Cluster State = 1
Waiting for cluster provisioning; attempt 1; sleeping for 0:02:30 seconds


[Ray on Vertex AI]: Cluster State = 1
Waiting for cluster provisioning; attempt 2; sleeping for 0:01:54.750000 seconds


[Ray on Vertex AI]: Cluster State = 1
Waiting for cluster provisioning; attempt 3; sleeping for 0:01:27.783750 seconds


...

[Ray on Vertex AI]: Cluster State = 3


2026-09-20 15:47:39,992	INFO dashboard_sdk.py:338 -- Uploading package gcs://_ray_pkg_1f7e7ecda0899134.zip.


2026-09-20 15:47:39,995	INFO packaging.py:588 -- Creating a file package for local module '/mnt/executor/scratch/scale-forecasting/src'.


[Ray on Vertex AI]: Successfully deleted the cluster.


ray ∥ bigquery run complete: nb04-ray-bq-1789918508-344fc058e7f8


## Review — both engines on one leaderboard

The stats models ran on Ray (`compute_engine='ray'`); the natives in BigQuery (`compute_engine='bigquery'`) — all under one `run_id`, ranked by `mean_wape`.

In [6]:
leaderboard(run_id, expect_models=cfg.models)

,model_type,compute_engine,n_cells,mean_wape,mean_mae
0,timesfm,bigquery,100,0.355866,11.472076
1,arima_plus,bigquery,100,0.384481,11.836770
2,holtwinters,ray,100,0.403176,14.368479
3,theta,ray,100,0.445700,15.222050


In [7]:
run_summary(run_id)

,run_id,created_at,status,python_runtime,n_series,n_models,backtest_on,runtime_seconds,total_wall_s,overhead_seconds,overhead_fraction,executor_instances,executor_cores,max_executors,executor_memory,executor_memory_overhead,dcu_milli_seconds,runtime_version,sizing,capacity
0,nb04-ray-bq-1789918508-344fc058e7f8,2026-09-20 15:35:20.502142+00:00,COMPLETED,ray,100,4,True,1100.220392,412.0,-688.220392,-1.670438,<NA>,<NA>,<NA>,None,None,<NA>,None,"{""statistical"":{""family"":""statistical"",""plans""...",None


## What the run recorded — the cluster sizing

The header's `job_telemetry` audits the sizing decision that actually ran: `runtime='ray'`, the autoscale spec (per-pool min/max) with the deterministic initial per-pool node counts, and (on the GPU path) the calibrated `sizing_gpu_fraction` + `accelerator_type`. On this CPU-only run `gpu_node_count` is `0` — the whole cluster is the CPU worker pool.

In [8]:
import json

hdr = _query(
    f"SELECT TO_JSON_STRING(job_telemetry) AS job_telemetry "
    f"FROM `{DATASET}.run_registry` WHERE run_id=@run_id",
    run_id,
)
json.loads(hdr["job_telemetry"].iloc[0]) if not hdr.empty else "(header not visible yet)"

{'accelerator_count': 1,
 'accelerator_type': 'NVIDIA_TESLA_T4',
 'autoscale': True,
 'cluster_name': 'sf-ray-nb04-ray-bq-1789918508-344fc058e7f8',
 'cpu_machine_type': 'n1-standard-8',
 'cpu_max_nodes': 4,
 'cpu_min_nodes': 1,
 'cpu_node_count': 4,
 'dashboard_address': '73f5040f225014a2-dot-us-central1.aiplatform-training.googleusercontent.com',
 'gpu_machine_type': 'n1-standard-8',
 'gpu_max_nodes': 2,
 'gpu_min_nodes': 1,
 'gpu_node_count': 0,
 'head_machine_type': 'n1-standard-16',
 'job_id': 'sf-nb04-ray-bq-1789918508-344fc058e7f8-statistical-a1',
 'job_status': 'SUCCEEDED',
 'n_cpu_cells': 200,
 'n_gpu_cells': 0,
 'python_version': '3.11',
 'ray_version': '2.47',
 'reuse': False,
 'runtime': 'ray',
 'scoring': {'hpo': 'off'},
 'sizing': {'statistical': {'family': 'statistical',
   'plans': [{'binding_axis': 'cores',
     'density_note': None,
     'derived_units': 4,
     'family': 'statistical',
     'max_units': 4,
     'min_units': 1,
     'n_cells': 200,
     'runtime': 'ray